In [1]:
import pandas as pd
import polars as pl

In [2]:
df = pl.read_csv('/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Data/complaints_2015to2018.csv')

In [3]:
df = df.drop_nulls(subset=['Consumer complaint narrative'])

In [4]:
df = df.drop(['Sub-product', 'Sub-issue', 'Company public response', 'Tags', 'Complaint ID', 'Submitted via'])

In [5]:
df.columns = ['Date', 'Product', 'Issue', 'Narrative', 'Company', 'State', 'ZIP', 'Date sent', 'Response', 'Timely Response']

In [6]:
df = df.with_columns(
    pl.col("Date").str.to_datetime(
        "%Y-%m-%dT%H:%M:%S%.fZ"
    )
)

df = df.with_columns(
    pl.col("Date sent").str.to_datetime(
        "%Y-%m-%dT%H:%M:%S%.fZ"
    )
)

In [7]:
df = df.sort(by='Date')

In [8]:
df = df.with_columns(
    (pl.col("Date sent") - pl.col("Date"))
    .dt.total_days()
    .alias("Delay")
)

In [9]:
df = df.with_columns(
    (pl.col("Timely Response") == "Yes").alias("Timely Response")
)

In [10]:
df = df.with_columns(
    pl.col("Product")
    .replace(
        {
            # Lending
            "Payday loan": "Lending",
            "Consumer Loan": "Lending",
            "Vehicle loan or lease": "Lending",
            "Student loan": "Lending",
            "Payday loan, title loan, or personal loan": "Lending",
            "Mortgage": "Lending",

            # Payments
            "Money transfers": "Payment & Money Services",
            "Virtual currency": "Payment & Money Services",
            "Money transfer, virtual currency, or money service": "Payment & Money Services",

            # Cards
            "Credit card": "Cards",
            "Prepaid card": "Cards",
            "Credit card or prepaid card": "Cards",

            # Accounts
            "Bank account or service": "Deposit Accounts",
            "Checking or savings account": "Deposit Accounts",

            # Credit Reporting
            "Credit reporting": "Credit Reporting",
            "Credit reporting, credit repair services, or other personal consumer reports": "Credit Reporting",

            # Debt Collection
            "Debt collection": "Debt Collection",

            # Other
            "Other financial service": "Other",
        }
    )
    .alias("Product")
)

In [11]:
issues = (
    df
    .select("Issue")
    .unique()
    .sort("Issue")
)[:, 0].to_list()

In [20]:
ISSUE_MAP = {

    # Fraud & Unauthorized Activity
    "Fraud or scam": "Fraud & Unauthorized Activity",
    "Identity theft / Fraud / Embezzlement": "Fraud & Unauthorized Activity",
    "Identity theft protection or other monitoring services": "Fraud & Unauthorized Activity",
    "Credit monitoring or identity theft protection services": "Fraud & Unauthorized Activity",
    "Credit monitoring or identity protection": "Fraud & Unauthorized Activity",
    "Problem with fraud alerts or security freezes": "Fraud & Unauthorized Activity",
    "Unauthorized transactions/trans. issues": "Fraud & Unauthorized Activity",
    "Unauthorized transactions or other transaction problem": "Fraud & Unauthorized Activity",
    "Lost or stolen check": "Fraud & Unauthorized Activity",
    "Lost or stolen money order": "Fraud & Unauthorized Activity",
    "Received a loan I didn't apply for": "Fraud & Unauthorized Activity",
    "Received a loan you didn't apply for": "Fraud & Unauthorized Activity",

    # Debt Collection
    "Attempts to collect debt not owed": "Debt Collection",
    "Cont'd attempts collect debt not owed": "Debt Collection",
    "Communication tactics": "Debt Collection",
    "Disclosure verification of debt": "Debt Collection",
    "False statements or representation": "Debt Collection",
    "Improper contact or sharing of info": "Debt Collection",
    "Taking/threatening an illegal action": "Debt Collection",
    "Threatened to contact someone or share information improperly": "Debt Collection",
    "Took or threatened to take negative or legal action": "Debt Collection",
    "Written notification about debt": "Debt Collection",
    "Sale of account": "Debt Collection",

    # Credit Reporting
    "Incorrect information on credit report": "Credit Reporting",
    "Incorrect information on your report": "Credit Reporting",
    "Credit reporting company's investigation": "Credit Reporting",
    "Problem with a credit reporting company's investigation into an existing problem": "Credit Reporting",
    "Improper use of my credit report": "Credit Reporting",
    "Improper use of your report": "Credit Reporting",
    "Unable to get credit report/credit score": "Credit Reporting",
    "Unable to get your credit report or credit score": "Credit Reporting",

    # Loan Servicing & Repayment
    "Loan servicing, payments, escrow account": "Loan Servicing & Repayment",
    "Dealing with my lender or servicer": "Loan Servicing & Repayment",
    "Dealing with your lender or servicer": "Loan Servicing & Repayment",
    "Loan payment wasn't credited to your account": "Loan Servicing & Repayment",
    "Payoff process": "Loan Servicing & Repayment",
    "Forbearance / Workout plans": "Loan Servicing & Repayment",
    "Problems at the end of the loan or lease": "Loan Servicing & Repayment",
    "Problem with the payoff process at the end of the loan": "Loan Servicing & Repayment",
    "Lender repossessed or sold the vehicle": "Loan Servicing & Repayment",
    "Vehicle was repossessed or sold the vehicle": "Loan Servicing & Repayment",
    "Property was sold": "Loan Servicing & Repayment",
    "Lender sold the property": "Loan Servicing & Repayment",
    "Can't repay my loan": "Loan Servicing & Repayment",
    "Struggling to pay mortgage": "Loan Servicing & Repayment",
    "Struggling to pay your loan": "Loan Servicing & Repayment",
    "Struggling to repay your loan": "Loan Servicing & Repayment",
    "Struggling to pay your bill": "Loan Servicing & Repayment",
    "Problems when you are unable to pay": "Loan Servicing & Repayment",
    "Delinquent account": "Loan Servicing & Repayment",
    "Bankruptcy": "Loan Servicing & Repayment",
    "Loan modification,collection,foreclosure": "Loan Servicing & Repayment",

    # Billing, Fees & Payments
    "APR or interest rate": "Billing, Fees & Payments",
    "Billing disputes": "Billing, Fees & Payments",
    "Billing statement": "Billing, Fees & Payments",
    "Fees": "Billing, Fees & Payments",
    "Fees or interest": "Billing, Fees & Payments",
    "Late fee": "Billing, Fees & Payments",
    "Excessive fees": "Billing, Fees & Payments",
    "Unexpected or other fees": "Billing, Fees & Payments",
    "Unexpected/Other fees": "Billing, Fees & Payments",
    "Charged fees or interest I didn't expect": "Billing, Fees & Payments",
    "Charged fees or interest you didn't expect": "Billing, Fees & Payments",
    "Making/receiving payments, sending money": "Billing, Fees & Payments",
    "Deposits and withdrawals": "Billing, Fees & Payments",
    "Problem when making payments": "Billing, Fees & Payments",
    "Trouble during payment process": "Billing, Fees & Payments",
    "Wrong amount charged or received": "Billing, Fees & Payments",
    "Money was taken from your bank account on the wrong day or for the wrong amount": "Billing, Fees & Payments",
    "Problem with a purchase or transfer": "Billing, Fees & Payments",
    "Transaction issue": "Billing, Fees & Payments",

    # Account Management
    "Managing an account": "Account Management",
    "Managing, opening, or closing account": "Account Management",
    "Opening an account": "Account Management",
    "Closing an account": "Account Management",
    "Closing your account": "Account Management",
    "Closing/Cancelling account": "Account Management",
    "Account opening, closing, or management": "Account Management",
    "Account terms and changes": "Account Management",
    "Credit limit changed": "Account Management",
    "Credit line increase/decrease": "Account Management",

    # Lending Process & Approval
    "Getting a loan": "Lending Process & Approval",
    "Getting a loan or lease": "Lending Process & Approval",
    "Getting the loan": "Lending Process & Approval",
    "Taking out the loan or lease": "Lending Process & Approval",
    "Application processing delay": "Lending Process & Approval",
    "Applied for loan/did not receive money": "Lending Process & Approval",
    "Was approved for a loan, but didn't receive money": "Lending Process & Approval",
    "Was approved for a loan, but didn't receive the money": "Lending Process & Approval",

    # Customer Service & Disclosures
    "Customer service / Customer relations": "Customer Service & Disclosures",
    "Customer service/Customer relations": "Customer Service & Disclosures",
    "Problem with customer service": "Customer Service & Disclosures",
    "Can't contact lender": "Customer Service & Disclosures",
    "Can't contact lender or servicer": "Customer Service & Disclosures",
    "Advertising": "Customer Service & Disclosures",
    "Advertising and marketing": "Customer Service & Disclosures",
    "Advertising and marketing, including promotional offers": "Customer Service & Disclosures",
    "Confusing or misleading advertising or marketing": "Customer Service & Disclosures",
    "Disclosures": "Customer Service & Disclosures",
    "Confusing or missing disclosures": "Customer Service & Disclosures",
}

In [21]:
df = df.with_columns(
    pl.col("Issue")
      .replace(ISSUE_MAP)
      .alias("Issue_Category")
)

df = df.with_columns(
    pl.when(pl.col("Issue_Category") == pl.col("Issue"))
      .then(pl.lit("Other"))
      .otherwise(pl.col("Issue_Category"))
      .alias("Issue_Category")
)

In [22]:
df = df.unique()

In [23]:
df.write_csv('/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Data/complaints_cleaned.csv')

In [19]:
df

Date,Product,Issue,Narrative,Company,State,ZIP,Date sent,Response,Timely Response,Delay,Issue_Category
datetime[μs],str,str,str,str,str,str,datetime[μs],str,bool,i64,str
2016-06-13 17:44:03,"""Lending""","""Charged fees or interest I did…","""On XXXX XXXX, 2016, I applied …","""Big Picture Loans, LLC""","""MD""","""20735""",2016-06-16 16:16:42,"""Closed with explanation""",true,2,"""Billing, Fees & Interest"""
2017-01-20 16:04:26,"""Lending""","""Dealing with my lender or serv…","""I was given bad advice from a …","""Navient Solutions, LLC.""","""NJ""","""080XX""",2017-01-20 16:04:27,"""Closed with explanation""",true,0,"""Loan Servicing"""
2015-10-24 15:23:25,"""Lending""","""Managing the loan or lease""","""In XXXX, my husband and I purc…","""BANK OF THE WEST""","""AZ""","""86442""",2015-10-24 15:23:26,"""Closed with explanation""",true,0,"""Other"""
2016-02-04 22:09:41,"""Debt Collection""","""Disclosure verification of deb…","""mrc receivables too me to cour…","""ENCORE CAPITAL GROUP INC.""","""MI""","""49221""",2016-02-04 22:09:42,"""Closed with explanation""",true,0,"""Debt Collection"""
2016-10-26 15:05:09,"""Cards""","""Billing disputes""","""A check was sent to the First …","""FIRSTBANK PUERTO RICO""","""GA""","""30809""",2016-10-28 13:37:37,"""Closed with explanation""",true,1,"""Billing, Fees & Interest"""
…,…,…,…,…,…,…,…,…,…,…,…
2018-04-13 16:57:26,"""Deposit Accounts""","""Problem with a lender or other…","""Suntrust Bank, located at XXXX…","""SUNTRUST BANKS, INC.""","""FL""","""33319""",2018-04-13 17:50:27,"""Closed with explanation""",true,0,"""Other"""
2016-10-07 13:17:23,"""Deposit Accounts""","""Problems caused by my funds be…","""On XXXX XXXX, 2016 I had a XXX…","""JPMORGAN CHASE & CO.""","""GA""","""30519""",2016-10-07 13:17:23,"""Closed with monetary relief""",true,0,"""Other"""
2017-04-12 19:39:49,"""Debt Collection""","""Cont'd attempts collect debt n…","""Enhanced Recovery Company a co…","""ERC""","""MN""","""55024""",2017-04-12 19:39:50,"""Closed with non-monetary relie…",true,0,"""Debt Collection"""
